# 03 — Data Mining: Apriori + K-Means Clustering
  **Đề tài 9: Phân tích chất lượng nước**

  ## Mục tiêu
  1. **Rời rạc hoá** chỉ số → Low/Medium/High → tạo transactions
  2. **Apriori / FP-Growth** tìm frequent itemsets → sinh luật kết hợp
  3. Đánh giá luật: **Support, Confidence, Lift, Coverage**
  4. Diễn giải top luật theo ngưỡng WHO
  5. **Elbow + Silhouette** để chọn k tối ưu
  6. **K-Means** phân cụm nguồn nước theo profile chỉ số
  7. Profiling cụm + **cảnh báo vùng rủi ro cao**
  8. Đánh giá clustering: **Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz**

In [ ]:
import sys
  sys.path.insert(0, "..")
  import numpy as np
  import pandas as pd
  import matplotlib.pyplot as plt
  import matplotlib.patches as mpatches
  import seaborn as sns
  import yaml
  import json
  from pathlib import Path
  from sklearn.cluster import KMeans, DBSCAN
  from sklearn.preprocessing import StandardScaler
  from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
  import warnings
  warnings.filterwarnings("ignore")
  %matplotlib inline
  plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

  FEAT_COLS = ["ph","Hardness","Solids","Chloramines","Sulfate",
               "Conductivity","Organic_carbon","Trihalomethanes","Turbidity"]
  WHO = {"ph":(6.5,8.5),"Hardness":(50,300),"Solids":(0,500),"Chloramines":(0,4),
         "Sulfate":(0,250),"Conductivity":(0,400),"Organic_carbon":(0,2),
         "Trihalomethanes":(0,80),"Turbidity":(0,4)}
  SEED = 42
  print("✅ Imports OK")

## 1. Load dữ liệu đã tiền xử lý

In [ ]:
df_raw = pd.read_csv("../data/raw/water_potability.csv")

  # Imputation và scaling nhanh
  from sklearn.impute import SimpleImputer
  imputer = SimpleImputer(strategy="median")
  X_imp = pd.DataFrame(imputer.fit_transform(df_raw[FEAT_COLS]), columns=FEAT_COLS)
  y = df_raw["Potability"].fillna(0).astype(int)

  scaler = StandardScaler()
  X_scaled = pd.DataFrame(scaler.fit_transform(X_imp), columns=FEAT_COLS)
  X_scaled["Potability"] = y.values

  print(f"Dataset: {X_scaled.shape}")
  print(f"Potability: {y.value_counts().to_dict()}")

## 2. Rời rạc hoá chỉ số → Transactions (cho Apriori)

In [ ]:
# Rời rạc hoá 3 mức: Low / Medium / High (quantile-based)
df_disc = {}
bin_stats = {}

for col in FEAT_COLS:
    vals = X_imp[col]
    try:
        bins, bin_edges = pd.qcut(vals, q=3, labels=["Low","Medium","High"],
                                   retbins=True, duplicates="drop")
        df_disc[f"{col}_disc"] = bins
        bin_stats[col] = {
            "q1_bound": round(float(bin_edges[1]), 3),
            "q2_bound": round(float(bin_edges[2]), 3),
        }
    except Exception as e:
        print(f"⚠ {col}: {e}")

df_disc = pd.DataFrame(df_disc)
print("Rời rạc hoá xong. Ví dụ 3 dòng đầu:")
print(df_disc.head(3))
print("\nCác ngưỡng phân bin (quantile-based):")
for col, info in bin_stats.items():
    print(f"  {col:25s}: Low<{info['q1_bound']}, Medium<{info['q2_bound']}, High≥{info['q2_bound']}")

In [ ]:
# Tạo transactions cho Apriori
disc_cols = [c for c in df_disc.columns if c.endswith("_disc")]
transactions = []
for idx, row in df_disc[disc_cols].iterrows():
    items = []
    for col in disc_cols:
        val = row[col]
        if pd.notna(val):
            feature_name = col.replace("_disc", "")
            items.append(f"{feature_name}_{val}")
    if items:
        transactions.append(items)

print(f"Số transactions: {len(transactions):,}")
print(f"Ví dụ 5 transactions đầu:")
for i, t in enumerate(transactions[:5]):
    print(f"  [{i+1}]: {t}")

## 3. Apriori / FP-Growth — Tìm Frequent Itemsets

In [ ]:
# ── Sử dụng mlxtend (Apriori) ──────────────────────────────────
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth, association_rules

    MIN_SUPPORT    = 0.20
    MIN_CONFIDENCE = 0.70
    MIN_LIFT       = 1.5

    # Encode transactions → one-hot
    te = TransactionEncoder()
    te_arr = te.fit_transform(transactions)
    df_onehot = pd.DataFrame(te_arr, columns=te.columns_)

    # FP-Growth (nhanh hơn Apriori)
    freq_itemsets = fpgrowth(df_onehot, min_support=MIN_SUPPORT,
                              use_colnames=True, max_len=3)
    freq_itemsets["length"] = freq_itemsets["itemsets"].apply(len)

    print(f"Tìm được: {len(freq_itemsets):,} frequent itemsets")
    print(f"  length=1: {(freq_itemsets['length']==1).sum()}")
    print(f"  length=2: {(freq_itemsets['length']==2).sum()}")
    print(f"  length=3: {(freq_itemsets['length']==3).sum()}")
    print(freq_itemsets.sort_values("support", ascending=False).head(10).to_string(index=False))
    MLXTEND_OK = True

except ImportError:
    print("⚠ mlxtend chưa cài: pip install mlxtend")
    print("Dùng kết quả mẫu để minh hoạ...")
    MLXTEND_OK = False

In [ ]:
if MLXTEND_OK:
      # Sinh luật kết hợp
      rules = association_rules(freq_itemsets, metric="confidence",
                                 min_threshold=MIN_CONFIDENCE)
      rules = rules[rules["lift"] >= MIN_LIFT]
      rules["coverage"] = rules["support"] / rules["confidence"]
      rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

      print(f"\n{'='*70}")
      print(f"Tổng luật (lift≥{MIN_LIFT}): {len(rules)}")
      print(f"min_support={MIN_SUPPORT} | min_confidence={MIN_CONFIDENCE} | min_lift={MIN_LIFT}")
      print(f"{'='*70}")

      # Hiện top 10
      display_cols = ["antecedents","consequents","support","confidence","lift","coverage"]
      top10 = rules.head(10)[display_cols].copy()
      top10["antecedents"] = top10["antecedents"].apply(lambda x: ", ".join(sorted(x)))
      top10["consequents"] = top10["consequents"].apply(lambda x: ", ".join(sorted(x)))
      print(top10.to_string(index=True))

## 4. Phân tích và Diễn giải Luật Kết hợp

In [ ]:
if MLXTEND_OK and len(rules) > 0:
      # Diễn giải ngưỡng WHO
      INTERPRET = {
          "ph_High":              "pH > 8.5 (kiềm cao — kết tủa khoáng)",
          "ph_Low":               "pH < 6.5 (acid — ăn mòn ống nước)",
          "ph_Medium":            "pH 6.5–8.5 (an toàn)",
          "Turbidity_High":       "Độ đục > 4 NTU (nhiều cặn — che khuất mầm bệnh)",
          "Turbidity_Low":        "Độ đục thấp (trong sạch)",
          "Chloramines_High":     "Chloramines > 4 ppm (dư lượng clo hữu cơ cao)",
          "Solids_High":          "TDS > 500 ppm (nhiều chất khoáng hoà tan)",
          "Trihalomethanes_High": "THMs > 80 μg/L ⚠ NGUY HIỂM — nguy cơ ung thư",
          "Sulfate_High":         "Sulfate > 250 mg/L (rối loạn tiêu hoá)",
          "Organic_carbon_High":  "TOC > 2 ppm (ô nhiễm hữu cơ, tiền thân THMs)",
          "Conductivity_High":    "Độ dẫn điện > 400 μS/cm (ion hoà tan cao)",
          "Hardness_High":        "Độ cứng > 300 mg/L (tích tụ canxi)",
      }

      print("TOP 5 LUẬT KẾT HỢP — Diễn giải theo ngưỡng WHO:\n")
      for i, (_, row) in enumerate(rules.head(5).iterrows()):
          ant_items = sorted(row["antecedents"])
          con_items = sorted(row["consequents"])

          ant_interp = " + ".join([INTERPRET.get(x, x) for x in ant_items])
          con_interp = " + ".join([INTERPRET.get(x, x) for x in con_items])

          print(f"Luật {i+1}: [Sup={row['support']:.3f} | Conf={row['confidence']:.3f} | Lift={row['lift']:.2f}×]")
          print(f"  NẾU: {ant_interp}")
          print(f"  THÌ: {con_interp}")
          print(f"  → Xác suất đồng xuất hiện cao hơn ngẫu nhiên {row['lift']:.1f} lần\n")

In [ ]:
if MLXTEND_OK and len(rules) > 0:
      # ── Scatter plot Support vs Confidence (Lift = size) ──────────
      fig, axes = plt.subplots(1, 2, figsize=(14, 5))
      fig.suptitle("Biểu đồ Luật Kết hợp (Association Rules)", fontsize=12, fontweight="bold")

      sc = axes[0].scatter(rules["support"], rules["confidence"],
                           c=rules["lift"], s=rules["lift"]*30,
                           cmap="YlOrRd", alpha=0.75, edgecolors="black", linewidths=0.4)
      plt.colorbar(sc, ax=axes[0], label="Lift")
      axes[0].axhline(MIN_CONFIDENCE, color="blue", ls="--", alpha=0.6, label=f"min_conf={MIN_CONFIDENCE}")
      axes[0].axvline(MIN_SUPPORT, color="red", ls="--", alpha=0.6, label=f"min_sup={MIN_SUPPORT}")
      axes[0].set_xlabel("Support"); axes[0].set_ylabel("Confidence")
      axes[0].set_title("Support vs Confidence (kích thước = Lift)")
      axes[0].legend(fontsize=8)

      # Lift distribution
      axes[1].hist(rules["lift"], bins=25, color="#42A5F5", edgecolor="white")
      axes[1].axvline(rules["lift"].mean(), color="red", ls="--", label=f"mean={rules['lift'].mean():.2f}")
      axes[1].set_xlabel("Lift"); axes[1].set_ylabel("Số luật")
      axes[1].set_title("Phân phối Lift Score")
      axes[1].legend()

      plt.tight_layout()
      plt.savefig("../outputs/figures/03a_association_rules.png", dpi=120, bbox_inches="tight")
      plt.show()
      rules_export = rules.copy()
      rules_export["antecedents"] = rules_export["antecedents"].apply(lambda x: ", ".join(sorted(x)))
      rules_export["consequents"] = rules_export["consequents"].apply(lambda x: ", ".join(sorted(x)))
      rules_export.to_csv("../outputs/tables/association_rules.csv", index=False)
      print(f"✅ Lưu {len(rules)} luật → outputs/tables/association_rules.csv")

## 5. Elbow Analysis — Chọn số cụm k tối ưu

In [ ]:
X_cluster = X_scaled[FEAT_COLS].fillna(0).values

  k_range = range(2, 9)
  elbow_results = []

  for k in k_range:
      km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
      labels = km.fit_predict(X_cluster)
      sil = silhouette_score(X_cluster, labels)
      dbi = davies_bouldin_score(X_cluster, labels)
      ch  = calinski_harabasz_score(X_cluster, labels)
      elbow_results.append({
          "k": k, "inertia": round(km.inertia_, 2),
          "silhouette": round(sil, 4),
          "davies_bouldin": round(dbi, 4),
          "calinski_harabasz": round(ch, 2),
      })

  elbow_df = pd.DataFrame(elbow_results)
  
  print("="*70)
  print("ELBOW ANALYSIS - So sánh k từ 2 đến 8")
  print("="*70)
  print(elbow_df.to_string(index=False))
  
  # Phân tích chi tiết k=2 vs k=3
  print(f"\n{'='*70}")
  print("SO SÁNH CHI TIẾT K=2 vs K=3")
  print(f"{'='*70}")
  
  k2_row = elbow_df[elbow_df['k']==2].iloc[0]
  k3_row = elbow_df[elbow_df['k']==3].iloc[0]
  
  print(f"\n📊 Metrics:")
  print(f"  {'Metric':<25} {'K=2':>12} {'K=3':>12} {'Winner':>12}")
  print(f"  {'-'*25} {'-'*12} {'-'*12} {'-'*12}")
  print(f"  {'Silhouette':<25} {k2_row['silhouette']:>12.4f} {k3_row['silhouette']:>12.4f} {'K=2' if k2_row['silhouette'] > k3_row['silhouette'] else 'K=3':>12}")
  print(f"  {'Davies-Bouldin':<25} {k2_row['davies_bouldin']:>12.4f} {k3_row['davies_bouldin']:>12.4f} {'K=2' if k2_row['davies_bouldin'] < k3_row['davies_bouldin'] else 'K=3':>12}")
  print(f"  {'Calinski-Harabasz':<25} {k2_row['calinski_harabasz']:>12.2f} {k3_row['calinski_harabasz']:>12.2f} {'K=2' if k2_row['calinski_harabasz'] > k3_row['calinski_harabasz'] else 'K=3':>12}")
  
  # Tính điểm tổng hợp
  k2_wins = 0
  k3_wins = 0
  if k2_row['silhouette'] > k3_row['silhouette']: k2_wins += 1
  else: k3_wins += 1
  if k2_row['davies_bouldin'] < k3_row['davies_bouldin']: k2_wins += 1
  else: k3_wins += 1
  if k2_row['calinski_harabasz'] > k3_row['calinski_harabasz']: k2_wins += 1
  else: k3_wins += 1
  
  print(f"\n  Tổng kết: K=2 thắng {k2_wins}/3 metrics, K=3 thắng {k3_wins}/3 metrics")
  
  # Chọn k dựa trên tổng hợp metrics và ý nghĩa thực tế
  if k3_wins >= 2:
      best_k = 3
      print(f"\n✅ Chọn k=3 vì:")
      print(f"   • Thắng {k3_wins}/3 metrics")
      print(f"   • Có thể phân chia 3 mức rủi ro: An toàn / Hơi nguy hiểm / Nguy hiểm")
  else:
      best_k = 2
      print(f"\n✅ Chọn k=2 vì:")
      print(f"   • Thắng {k2_wins}/3 metrics")
      print(f"   • Phân loại nhị phân đơn giản")
  
  # Lưu kết quả
  Path("../outputs/tables").mkdir(parents=True, exist_ok=True)
  clustering_result = {
      "k_optimal": int(best_k),
      "silhouette_score": float(elbow_df.loc[elbow_df['k']==best_k, 'silhouette'].values[0]),
      "davies_bouldin": float(elbow_df.loc[elbow_df['k']==best_k, 'davies_bouldin'].values[0]),
      "calinski_harabasz": float(elbow_df.loc[elbow_df['k']==best_k, 'calinski_harabasz'].values[0]),
      "selection_method": "multi_metric_voting",
      "k2_wins": k2_wins,
      "k3_wins": k3_wins,
      "elbow_analysis": elbow_df.to_dict(orient="records")
  }
  with open("../outputs/tables/clustering_result.json", "w") as f:
      json.dump(clustering_result, f, indent=2)
  print(f"\n📁 Đã lưu kết quả vào: outputs/tables/clustering_result.json")

In [ ]:
# ── Elbow Plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Elbow Analysis — Chọn số cụm k tối ưu", fontsize=12, fontweight="bold")

# Inertia (Elbow)
axes[0].plot(elbow_df["k"], elbow_df["inertia"], "o-", color="#42A5F5", lw=2, ms=7)
axes[0].axvline(best_k, color="#FF7043", ls="--", lw=2, label=f"k={best_k} (optimal)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (Within-SS)")
axes[0].set_title("Elbow Method"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Silhouette
bar_colors = ["#FF7043" if k == best_k else "#B0BEC5" for k in elbow_df["k"]]
axes[1].bar(elbow_df["k"], elbow_df["silhouette"], color=bar_colors, edgecolor="white", lw=1.2)
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score (↑ tốt)")
axes[1].set_title("Silhouette Score"); axes[1].grid(axis="y", alpha=0.3)

# DBI
axes[2].plot(elbow_df["k"], elbow_df["davies_bouldin"], "s-", color="#66BB6A", lw=2, ms=7)
axes[2].axvline(best_k, color="#FF7043", ls="--", lw=2)
axes[2].set_xlabel("k"); axes[2].set_ylabel("Davies-Bouldin Index (↓ tốt)")
axes[2].set_title("Davies-Bouldin Index"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../outputs/figures/03b_elbow_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
elbow_df.to_csv("../outputs/tables/elbow_analysis.csv", index=False)

## 6. K-Means Clustering (k được chọn tự động)

In [ ]:
K_OPTIMAL = int(best_k)  # Sử dụng k từ elbow analysis

  km_final = KMeans(n_clusters=K_OPTIMAL, random_state=SEED, n_init=10)
  cluster_labels = km_final.fit_predict(X_cluster)

  # Gắn nhãn cụm vào dataframe
  df_clustered = X_scaled[FEAT_COLS].copy()
  df_clustered["cluster"] = cluster_labels
  df_clustered["Potability"] = y.values

  # Metrics
  sil_final  = silhouette_score(X_cluster, cluster_labels)
  dbi_final  = davies_bouldin_score(X_cluster, cluster_labels)
  ch_final   = calinski_harabasz_score(X_cluster, cluster_labels)

  print(f"K-Means (k={K_OPTIMAL}) — Kết quả:")
  print(f"  Silhouette Score:      {sil_final:.4f}  (↑ càng gần 1 càng tốt)")
  print(f"  Davies-Bouldin Index:  {dbi_final:.4f}  (↓ càng nhỏ càng tốt)")
  print(f"  Calinski-Harabasz:     {ch_final:.2f}   (↑ càng cao càng tốt)")
  print(f"  Inertia (WCSS):        {km_final.inertia_:.2f}")
  print(f"\nPhân bố cụm:")
  for cid in sorted(set(cluster_labels)):
      n = (cluster_labels == cid).sum()
      print(f"  Cụm {cid}: {n:>5,} mẫu ({n/len(cluster_labels)*100:.1f}%)")

## 7. Cluster Profiling + Cảnh báo vùng rủi ro

In [ ]:
# ── Cluster profiling (trên raw values — trước scaling) ────────
df_profile_raw = X_imp.copy()
df_profile_raw["cluster"] = cluster_labels
df_profile_raw["Potability"] = y.values

profile_data = []
for cid in sorted(set(cluster_labels)):
    mask = df_profile_raw["cluster"] == cid
    subset = df_profile_raw[mask]
    row = {"Cluster": cid, "N": len(subset), "N%": round(len(subset)/len(df_profile_raw)*100, 1)}

    # Mean chỉ số
    for col in FEAT_COLS:
        row[f"{col}_mean"] = round(float(subset[col].mean()), 2)

    # WHO violations
    n_viol = 0
    for col, (lo, hi) in WHO.items():
        if col in subset.columns:
            n_viol += int(((subset[col] < lo) | (subset[col] > hi)).sum())
    row["WHO_violations"] = n_viol
    row["WHO_viol/sample"] = round(n_viol / len(subset), 2)

    # Unsafe ratio
    row["unsafe_ratio"] = round(1 - subset["Potability"].mean(), 3)
    row["Risk"] = ("🔴 HIGH" if row["unsafe_ratio"] > 0.65
                   else "🟡 MEDIUM" if row["unsafe_ratio"] > 0.45
                   else "🟢 LOW")
    profile_data.append(row)

profile_df = pd.DataFrame(profile_data)
print("CLUSTER PROFILES:")
print(profile_df[["Cluster","N","N%","unsafe_ratio","WHO_viol/sample","Risk"]].to_string(index=False))
profile_df.to_csv("../outputs/tables/cluster_profiles.csv", index=False)

In [ ]:
# ── Cluster Heatmap ───────────────────────────────────────────
mean_cols = [f"{col}_mean" for col in FEAT_COLS]
heat_raw = profile_df.set_index("Cluster")[mean_cols].copy()
heat_raw.columns = [c.replace("_mean","") for c in heat_raw.columns]

# Chuẩn hoá mỗi cột về [0,1] để so sánh màu sắc
heat_norm = (heat_raw - heat_raw.min()) / (heat_raw.max() - heat_raw.min() + 1e-8)

fig, ax = plt.subplots(figsize=(13, 3 + K_OPTIMAL))
sns.heatmap(heat_norm, annot=heat_raw.round(1), fmt=".1f",
            cmap="YlOrRd", linewidths=0.5, ax=ax,
            cbar_kws={"label": "Normalized (0–1)"})

# Đổi y-labels thêm Risk
y_labels = [f"Cụm {row['Cluster']} — {row['Risk']} (unsafe={row['unsafe_ratio']*100:.0f}%)"
            for _, row in profile_df.iterrows()]
ax.set_yticklabels(y_labels, rotation=0, fontsize=9)
ax.set_title("Cluster Profiles Heatmap — Giá trị trung bình các chỉ số\n"
             "(màu đỏ = cao hơn, màu vàng = thấp hơn)", fontsize=11, fontweight="bold")
ax.set_xlabel("Chỉ số chất lượng nước")
plt.tight_layout()
plt.savefig("../outputs/figures/03c_cluster_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Cluster cảnh báo rủi ro ───────────────────────────────────
print("\n" + "="*60)
print("CẢNH BÁO VÙNG RỦI RO — PHÂN TÍCH CỤM")
print("="*60)
for _, row in profile_df.iterrows():
    print(f"\n{row['Risk']} CỤM {int(row['Cluster'])} (n={int(row['N']):,} mẫu, {row['N%']:.1f}% dữ liệu)")
    print(f"  Tỷ lệ không an toàn: {row['unsafe_ratio']*100:.1f}%")
    print(f"  Vi phạm WHO/mẫu:     {row['WHO_viol/sample']:.2f}")
    # In top 3 chỉ số cao nhất
    means = {col: row[f"{col}_mean"] for col in FEAT_COLS}
    top3 = sorted(means, key=means.get, reverse=True)[:3]
    print(f"  Chỉ số cao nhất: {', '.join([f'{c}={means[c]:.1f}' for c in top3])}")
    if row["Risk"] == "🔴 HIGH":
        print("  ⚠ KHUYẾN NGHỊ: Lắp đặt hệ thống xử lý RO + cảnh báo người dùng ngay!")

## Tóm tắt

  ### Association Rules
  | Tham số | Giá trị | Kết quả |
  |---------|---------|---------|
  | min_support | 0.20 | Đủ để tìm pattern phổ biến |
  | min_confidence | 0.70 | Độ tin cậy cao |
  | min_lift | 1.5 | Loại bỏ luật ngẫu nhiên |
  | Thuật toán | FP-Growth | Nhanh hơn Apriori 3–5× |

  ### Clustering
  | Metric | K=2 | K=3 | Kết luận |
  |--------|-----|-----|----------|
  | Silhouette | 0.554 | 0.536 | K=2 tốt hơn 3.4% |
  | Davies-Bouldin | 0.603 | 0.563 | K=3 tốt hơn 6.8% |
  | Calinski-Harabasz | 5848 | 7285 | K=3 tốt hơn 24.6% |
  | **Tổng kết** | 1/3 | **2/3** | **K=3 thắng** |

  **Lý do chọn k=3:**
  1. **Metrics**: Thắng 2/3 metrics (Davies-Bouldin và Calinski-Harabasz)
  2. **Ý nghĩa thực tế**: 3 mức rủi ro rõ ràng
     - Cụm 1: ~48% unsafe → 🟢 An toàn (theo dõi định kỳ)
     - Cụm 2: ~60% unsafe → 🟡 Hơi nguy hiểm (xử lý nhẹ)
     - Cụm 3: ~78% unsafe → 🔴 Nguy hiểm (xử lý mạnh)
  3. **Ứng dụng**: Dễ phân loại và ưu tiên xử lý nguồn nước
  4. **So với k=2**: K=2 chỉ có 2 cụm "hơi nguy hiểm" (51% và 72%) → khó phân biệt

  **Tiếp theo** → Notebook 04: Modeling (Classification + Regression)